# 基礎編8
このノートブックでは、コントラクト内蔵キーバリューストアを使った基本的な例を説明します。

## 内蔵キーバリューストアの特徴

- スマートコントラクトに内蔵されている。
- それぞれのスマートコントラクト内に隔離されている。
- スマートコントラクトAPIで操作する。

## APIの説明  

- keyValueGetでkeyに割り当てられた値を取得します。  
- keyValueAddでkeyに割り当てられた値にvalueを加算します。
- keyValueSetでkeyに割り当てられた値を代入します。
- keyValueKeysでキーの列挙を返します。列挙の順序は登録順です。(chronological order)
- keyValueDeleteで指定されたキーと割り当てられた値を削除します。
- keyValueDeleteAllですべてのキーと割り当てられた値を削除します。

## 準備

In [1]:
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

## スマートコントラクトのデプロイ

内蔵キーバリューストアの操作の説明用のスマートコントラクトを作成します。  
このコントラクトは、指定されたfuncに従って、あらかじめ決まった動作を行います。
（このコントラクトの動作は説明用に単純化しており、深い意味はありません）

In [2]:
var cid = await deploySmartContract({ func: 'string' }, function basic8(func){
    switch(func){
    case "取得":
        return {
            value1: keyValueGet("key1"),
            value2: keyValueGet("key2"),
        };
    case "代入":           
        keyValueSet("key1",111);
        keyValueSet("key2",222);
        return;
    case "加算":
        keyValueAdd("key1",10);
        keyValueAdd("key2",100);
        return;  
    case "列挙":            
        var result = [];
        var e = keyValueKeys();
        for(var key = e.next(); key !== null; key = e.next()){
            var value = keyValueGet(key);
            result.push(key+":"+value);
        }
        return result;
    case "削除":            
        keyValueDelete("key1");
        return;
    case "全削除":            
        keyValueDeleteAll();
        return;
    }
});

## 実行例―取得/代入

初期状態ではキーバリューストアには何も格納されていません。  
値を取得するとnullが返ります。

In [3]:
var resp = await rpc.call(adminWallet, cid, { func: "取得" });
console.log(resp);

{
  txno: 701758,
  txid: 'xwSCzARHgQVPhsxtfYs6qSVdJ27urR5ztpCX9cQneebpLB',
  status: 'ok',
  value: { value1: null, value2: null }
}


キーに値を代入します。

In [4]:
var resp = await rpc.call(adminWallet, cid, { func: "代入" });
console.log(resp);

{
  txno: 701759,
  txid: 'xGELrihyjtZHyoqxnQReMHqdwzKhSh5ZR5YuArWJQMw98',
  status: 'ok',
  value: null
}


確認のため、キーの値を取得します。  
スマートコントラクト内で代入した値が返ります。

In [5]:
var resp = await rpc.call(adminWallet, cid, { func: "取得" });
console.log(resp);

{
  txno: 701760,
  txid: 'xBJNMEY2rkppLs5eEtuWmiNzzTbUqJgAVhS9KtSN69hF7',
  status: 'ok',
  value: { value1: 111, value2: 222 }
}


## 実行例―加算

キーに値を加算します。

In [6]:
var resp = await rpc.call(adminWallet, cid, { func: "加算" });
console.log(resp);

{
  txno: 701761,
  txid: 'xjw4gEtMbR8wqc7oACqiPDqgEgxi2XsW9Fn7TsYnbUZJq',
  status: 'ok',
  value: null
}


確認のため、キーの値を取得します。  
スマートコントラクト内で加算された結果の値が返ります。

In [7]:
var resp = await rpc.call(adminWallet, cid, { func: "取得" });
console.log(resp);

{
  txno: 701762,
  txid: 'xsKgtEN99boDKxzw9izJmPMcoENbFphbHGFD4W2oEuQCQB',
  status: 'ok',
  value: { value1: 121, value2: 322 }
}


キーに値をもういちど加算します。

In [8]:
var resp = await rpc.call(adminWallet, cid, { func: "加算" });
console.log(resp);

{
  txno: 701763,
  txid: 'xcLGQo4rmcjU8H2fLtWpwaSwzrfkk758pbvU7fkGKgQJGB',
  status: 'ok',
  value: null
}


確認のため、キーの値を取得します。  
さらに加算された結果の値が返ります。

In [9]:
var resp = await rpc.call(adminWallet, cid, { func: "取得" });
console.log(resp);

{
  txno: 701764,
  txid: 'xfNJdKRiPhvnsMvS83mk9SsaYR9ZxfeRJmQ8FjPQV9iyMB',
  status: 'ok',
  value: { value1: 131, value2: 422 }
}


## 実行例―列挙

キーバリューストアに格納されているキーとバリューを列挙します。

In [10]:
var resp = await rpc.call(adminWallet, cid, { func: "列挙" });
console.log(resp);

{
  txno: 701765,
  txid: 'xLgg3jbCT3sY2dbH887zShCBsYqCBfbqZabfW6n2xGSdLB',
  status: 'ok',
  value: [ 'key1:131', 'key2:422' ]
}


## 実行例―削除

キーバリューストアに格納されているキーとバリューを削除します。

In [11]:
var resp = await rpc.call(adminWallet, cid, { func: "削除" });
console.log(resp);

{
  txno: 701766,
  txid: 'xVAj97mhwAimvfzZHsaRijbr7oeJmEx39Dhyt8F4TXXwn',
  status: 'ok',
  value: null
}


確認のため、キーバリューを列挙します。  
key1が削除されています。

In [12]:
var resp = await rpc.call(adminWallet, cid, { func: "列挙" });
console.log(resp);

{
  txno: 701767,
  txid: 'xpHPnCxwBNk5HLvyVUPfAzAfjcaVDs46h63FDzFswX9wJB',
  status: 'ok',
  value: [ 'key2:422' ]
}


キーに値を再度代入します。

In [13]:
var resp = await rpc.call(adminWallet, cid, { func: "代入" });
console.log(resp);

{
  txno: 701768,
  txid: 'x2nzm4RGTeWvCC5qGbk3razJUBQYR8ZEksPtRZKK58a4QB',
  status: 'ok',
  value: null
}


確認のため、キーバリューを列挙します。  
key1が復活します。

In [14]:
var resp = await rpc.call(adminWallet, cid, { func: "列挙" });
console.log(resp);

{
  txno: 701769,
  txid: 'x8q49cMMJWci79LByiiLza57z8F8zvVv66BgRHayiJUoMB',
  status: 'ok',
  value: [ 'key2:222', 'key1:111' ]
}


## 実行例―全削除

キーバリューストアに格納されているキーとバリューをすべて削除します。

In [15]:
var resp = await rpc.call(adminWallet, cid, { func: "全削除" });
console.log(resp);

{
  txno: 701770,
  txid: 'x9RcgsJZR9sCLdCTLrpK8A9zmehVHj9doHGer3mJKAVuDB',
  status: 'ok',
  value: null
}


確認のため、キーバリューを列挙します。  

In [16]:
var resp = await rpc.call(adminWallet, cid, { func: "列挙" });
console.log(resp);

{
  txno: 701771,
  txid: 'xxd8wiYQXcA3hoLqdiioaq3vXwFTaUh56FsWZpNmidxj7',
  status: 'ok',
  value: []
}
